# Trial-Level Fiber Photometry Analysis

This notebook uses cue-related licking classifications generated during preprocessing to compare trial types within one processed photometry session.

It covers cue-lick versus miss trials, mutually exclusive licking categories, trial heatmaps, trial-local z-scoring, and comparison of IRLS dF/F with raw 465 fractional fluorescence.

## 1. Configuration

Set the repository, photometry root, and mouse/date/run. The processed-session filename is assembled automatically.

In [ ]:
from pathlib import Path

PHOTOMETRY_ROOT = Path(r"Z:\Photometry")

MOUSE = "DK21"
DATE = "230704"
RUN = 2
CHANNEL = 1

WINDOW_PRE = 5.0
WINDOW_POST = 10.0
BASELINE_START = -5.0
BASELINE_END = 0.0

SESSION_DIR = PHOTOMETRY_ROOT / MOUSE / f"{MOUSE}_{DATE}"
PROCESSED_SESSION = SESSION_DIR / f"{MOUSE}-{DATE}-{RUN:03d}-processed.npz"

print("Processed session:", PROCESSED_SESSION)

In [ ]:
# ============================================================
# Find repository root and import project modules
# ============================================================

from pathlib import Path
import sys

current_dir = Path.cwd()

PROJECT_ROOT = None

for candidate in [current_dir, *current_dir.parents]:
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find repository root containing 'src'."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src import save_sessiondata
from src import pynapple_utils

print("Repository:")
print(PROJECT_ROOT)

print("\nProcessed session:")
print(PROCESSED_SESSION)

## 2. Load the processed session

Trial classifications were created during preprocessing using each trial's actual cue onset and cue offset. The post-cue response window is the 2 seconds immediately following cue offset.

In [ ]:
session = save_sessiondata.load_session(
    PROCESSED_SESSION
)

data = pynapple_utils.session_to_pynapple(
    session
)

print("Mouse:", session["mouse"])
print("Date:", session["date"])
print("Run:", session["run"])
print("Cue trials:", len(session["cue_onset"]))

## 3. Trial classifications

Saved classifications include `cue_lick`, `post_cue_lick`, `cue_only`, `post_only`, `cue_and_post`, and `cue_miss`.

In [ ]:
cue_onset = np.asarray(session["cue_onset"], dtype=float)
cue_offset = np.asarray(session["cue_offset"], dtype=float)
cue_lick = np.asarray(session["cue_lick"], dtype=bool)
post_cue_lick = np.asarray(session["post_cue_lick"], dtype=bool)
cue_only = np.asarray(session["cue_only"], dtype=bool)
post_only = np.asarray(session["post_only"], dtype=bool)
cue_and_post = np.asarray(session["cue_and_post"], dtype=bool)
cue_miss = np.asarray(session["cue_miss"], dtype=bool)

print("Total:", len(cue_onset))
print("Cue lick:", cue_lick.sum())
print("Post-cue lick:", post_cue_lick.sum())
print("Cue only:", cue_only.sum())
print("Post only:", post_only.sum())
print("Cue + post:", cue_and_post.sum())
print("Miss:", cue_miss.sum())
print("Exclusive-category total:", cue_only.sum()+post_only.sum()+cue_and_post.sum()+cue_miss.sum())

## 4. Photometry representations

Both IRLS-corrected dF/F and raw 465 fractional fluorescence are retained. IRLS dF/F is one available representation and is not assumed to be optimal for every sensor/session.

In [ ]:
dff_time = np.asarray(data[f"dff_ch{CHANNEL}"].t, dtype=float)
dff = np.asarray(data[f"dff_ch{CHANNEL}"].d, dtype=float)

raw_465_time = np.asarray(session[f"photo_time_465_ch{CHANNEL}"], dtype=float)
raw_465 = np.asarray(session[f"photometry_465_ch{CHANNEL}"], dtype=float)
f0 = np.nanmedian(raw_465)
raw_465_fractional = (raw_465 - f0) / f0

print("dF/F samples:", len(dff))
print("Raw 465 samples:", len(raw_465))

## 5. Trial-alignment and normalization helpers

In [ ]:
def event_aligned_traces(signal_time, signal, event_times, pre=5.0, post=10.0):
    signal_time = np.asarray(signal_time, dtype=float)
    signal = np.asarray(signal, dtype=float)
    event_times = np.asarray(event_times, dtype=float)
    dt = np.median(np.diff(signal_time))
    relative_time = np.arange(-pre, post + 0.5*dt, dt)

    valid_indices = np.asarray([
        i for i, event in enumerate(event_times)
        if event-pre >= signal_time[0] and event+post <= signal_time[-1]
    ], dtype=int)

    traces = np.full((len(valid_indices), len(relative_time)), np.nan)
    for row, i in enumerate(valid_indices):
        traces[row] = np.interp(event_times[i] + relative_time, signal_time, signal)

    return relative_time, traces, valid_indices


def mean_and_sem(traces):
    if traces.shape[0] == 0:
        empty = np.full(traces.shape[1], np.nan)
        return empty, empty.copy()
    mean = np.nanmean(traces, axis=0)
    sem = (np.nanstd(traces, axis=0, ddof=1) / np.sqrt(traces.shape[0])
           if traces.shape[0] > 1 else np.full(traces.shape[1], np.nan))
    return mean, sem


def baseline_zscore_trials(relative_time, traces, baseline_start=-5.0, baseline_end=0.0):
    mask = (relative_time >= baseline_start) & (relative_time < baseline_end)
    if not np.any(mask):
        raise ValueError("Baseline window contains no samples.")

    output = np.full_like(traces, np.nan, dtype=float)
    for i in range(traces.shape[0]):
        baseline = traces[i, mask]
        mu = np.nanmean(baseline)
        sd = np.nanstd(baseline, ddof=1)
        if np.isfinite(sd) and sd > 0:
            output[i] = (traces[i] - mu) / sd
    return output

## 6. Align all cue trials

In [ ]:
cue_time, cue_dff, valid_idx = event_aligned_traces(
    dff_time, dff, cue_onset, WINDOW_PRE, WINDOW_POST
)

v_cue_lick = cue_lick[valid_idx]
v_cue_only = cue_only[valid_idx]
v_post_only = post_only[valid_idx]
v_cue_and_post = cue_and_post[valid_idx]
v_miss = cue_miss[valid_idx]

print("Complete cue trials:", cue_dff.shape[0])
print("Samples/trial:", cue_dff.shape[1])

## 7. Cue-lick versus miss

`cue_lick` includes any trial with licking during the cue. `cue_miss` contains trials with no licking during either the cue or the post-cue response window. Post-only trials are therefore not part of this comparison.

In [ ]:
cue_lick_traces = cue_dff[v_cue_lick]
miss_traces = cue_dff[v_miss]

cue_lick_mean, cue_lick_sem = mean_and_sem(cue_lick_traces)
miss_mean, miss_sem = mean_and_sem(miss_traces)

print("Cue-lick trials:", len(cue_lick_traces))
print("Miss trials:", len(miss_traces))

plt.figure(figsize=(9,5))
plt.plot(cue_time, cue_lick_mean, linewidth=2, label=f"Cue lick (n={len(cue_lick_traces)})")
plt.fill_between(cue_time, cue_lick_mean-cue_lick_sem, cue_lick_mean+cue_lick_sem, alpha=.2)
plt.plot(cue_time, miss_mean, linewidth=2, label=f"Miss (n={len(miss_traces)})")
plt.fill_between(cue_time, miss_mean-miss_sem, miss_mean+miss_sem, alpha=.2)
plt.axvline(0, linestyle="--"); plt.axhline(0, linestyle=":")
plt.xlabel("Time from cue onset (s)"); plt.ylabel("dF/F")
plt.title("Cue licking versus miss trials"); plt.legend()
plt.tight_layout(); plt.show()

## 8. Mutually exclusive trial categories

In [ ]:
trial_groups = {
    "Cue only": cue_dff[v_cue_only],
    "Post only": cue_dff[v_post_only],
    "Cue + post": cue_dff[v_cue_and_post],
    "Miss": cue_dff[v_miss],
}

for name, traces in trial_groups.items():
    print(f"{name:12s}: {traces.shape[0]}")

plt.figure(figsize=(10,6))
for name, traces in trial_groups.items():
    if len(traces):
        mean, sem = mean_and_sem(traces)
        plt.plot(cue_time, mean, linewidth=2, label=f"{name} (n={len(traces)})")
plt.axvline(0, linestyle="--"); plt.axhline(0, linestyle=":")
plt.xlabel("Time from cue onset (s)"); plt.ylabel("dF/F")
plt.title("Cue-aligned photometry by trial category"); plt.legend()
plt.tight_layout(); plt.show()

## 9. Trial heatmaps

In [ ]:
def plot_trial_heatmap(time, traces, title, label="dF/F"):
    if len(traces) == 0:
        print("No trials:", title)
        return
    plt.figure(figsize=(10,6))
    plt.imshow(traces, aspect="auto",
               extent=[time[0], time[-1], traces.shape[0], 0],
               interpolation="none")
    plt.axvline(0, linestyle="--")
    plt.xlabel("Time from cue onset (s)"); plt.ylabel("Trial")
    plt.title(title); plt.colorbar(label=label)
    plt.tight_layout(); plt.show()

plot_trial_heatmap(cue_time, cue_lick_traces, "Cue-lick trials")
plot_trial_heatmap(cue_time, miss_traces, "Cue-miss trials")

## 10. Trial-local z-scoring

Each trial is normalized to its own pre-cue baseline using `z = (trial - baseline mean) / baseline SD`.

In [ ]:
cue_dff_z = baseline_zscore_trials(
    cue_time, cue_dff, BASELINE_START, BASELINE_END
)
cue_lick_z = cue_dff_z[v_cue_lick]
miss_z = cue_dff_z[v_miss]

a, a_sem = mean_and_sem(cue_lick_z)
b, b_sem = mean_and_sem(miss_z)

plt.figure(figsize=(9,5))
plt.plot(cue_time, a, linewidth=2, label="Cue lick")
plt.fill_between(cue_time, a-a_sem, a+a_sem, alpha=.2)
plt.plot(cue_time, b, linewidth=2, label="Miss")
plt.fill_between(cue_time, b-b_sem, b+b_sem, alpha=.2)
plt.axvline(0, linestyle="--"); plt.axhline(0, linestyle=":")
plt.xlabel("Time from cue onset (s)"); plt.ylabel("Trial-baseline z-score")
plt.title("Cue-lick versus miss: trial-local dF/F z-score"); plt.legend()
plt.tight_layout(); plt.show()

## 11. Raw 465 comparison

Repeat the same trial alignment using raw 465 fractional fluorescence to assess whether IRLS reference correction changes the apparent trial response.

In [ ]:
raw_time, cue_raw, raw_valid_idx = event_aligned_traces(
    raw_465_time, raw_465_fractional, cue_onset, WINDOW_PRE, WINDOW_POST
)

print("Same valid trials:", np.array_equal(valid_idx, raw_valid_idx))

raw_cue_lick = cue_raw[cue_lick[raw_valid_idx]]
raw_miss = cue_raw[cue_miss[raw_valid_idx]]
raw_a, raw_a_sem = mean_and_sem(raw_cue_lick)
raw_b, raw_b_sem = mean_and_sem(raw_miss)

fig, axes = plt.subplots(2,1,figsize=(9,8),sharex=True)
axes[0].plot(cue_time, cue_lick_mean, linewidth=2, label="Cue lick")
axes[0].plot(cue_time, miss_mean, linewidth=2, label="Miss")
axes[0].axvline(0, linestyle="--"); axes[0].set_ylabel("dF/F")
axes[0].set_title("IRLS-corrected dF/F"); axes[0].legend()

axes[1].plot(raw_time, raw_a, linewidth=2, label="Cue lick")
axes[1].plot(raw_time, raw_b, linewidth=2, label="Miss")
axes[1].axvline(0, linestyle="--"); axes[1].set_xlabel("Time from cue onset (s)")
axes[1].set_ylabel("Fractional 465"); axes[1].set_title("Raw 465")
axes[1].legend()
plt.tight_layout(); plt.show()

## 12. Raw 465 trial-local z-score

In [ ]:
cue_raw_z = baseline_zscore_trials(
    raw_time, cue_raw, BASELINE_START, BASELINE_END
)
raw_cue_lick_z = cue_raw_z[cue_lick[raw_valid_idx]]
raw_miss_z = cue_raw_z[cue_miss[raw_valid_idx]]
ra, ra_sem = mean_and_sem(raw_cue_lick_z)
rb, rb_sem = mean_and_sem(raw_miss_z)

plt.figure(figsize=(9,5))
plt.plot(raw_time, ra, linewidth=2, label="Cue lick")
plt.fill_between(raw_time, ra-ra_sem, ra+ra_sem, alpha=.2)
plt.plot(raw_time, rb, linewidth=2, label="Miss")
plt.fill_between(raw_time, rb-rb_sem, rb+rb_sem, alpha=.2)
plt.axvline(0, linestyle="--"); plt.axhline(0, linestyle=":")
plt.xlabel("Time from cue onset (s)"); plt.ylabel("Raw 465 trial-baseline z-score")
plt.title("Raw 465: cue-lick versus miss"); plt.legend()
plt.tight_layout(); plt.show()

## 13. Interpretation and next steps

This notebook analyzes trial variability within one mouse/session.

Important points:

- trial classifications come from preprocessing
- actual cue onset and offset are used
- post-cue licking is defined relative to cue offset
- IRLS dF/F and raw 465 remain alternative representations
- trial-local z-scoring provides an additional normalization
- individual trials should not be treated as independent mice in group statistics

For group analysis, calculate session/mouse-level traces first and then combine those mouse-level summaries across animals.